# Removing and steering the hallucination direction · Llama-3.1-8B and Aya Expanse 8B on Hindi

Is there a direction in the residual stream that the model **uses** to hallucinate, or only one a probe can **read**?
Each model goes through the stages below in turn. Only one model is on the GPUs at a time.

1. **Find it.** The hallucination direction is the difference of means between hallucinated and supported entity tokens
   (`mean(Not Supported / Insufficient) − mean(Supported)`, as ARENA's `MMProbe`), in our 60 judge-annotated Hindi answers, at every layer.
   It is cross-fitted: fitted on one half of the answers (2 folds by prompt, as in the probe notebooks) and scored on the other half.
   The layer `L*` with the best held-out span-max AUC is used. As in SafeTune's selection sweep, the first and last 20% of layers are excluded.
2. **Locate it** with SafeTune `identify_safety_neurons` (weight mode). It ranks the MLP neurons whose `down_proj` column writes most strongly along the direction, layer by layer.
   **Heads:** for every attention head, three things are saved per judged entity (span): how much its output weights align with the direction, how much it actually writes along the direction,
   and where it attends (question, answer so far, the first token, and the entropy of its attention). Attention patterns of the top heads are saved for a few answers too.
   Nothing is reduced to a figure only: every number is saved, so heatmaps and correlation matrices can be remade later.
3. **Remove / steer it on the existing answers** with SafeTune `RefusalDirectionModel`. Despite the name, it takes any direction.
   `ablate` removes the direction at every layer. `steer` adds `α × (mean difference)` at `L*`.
   The answers are read word for word (teacher-forced), always with the fold's own held-out direction. Measures:
   - **Probability shift:** the change in the model's log-probability of hallucinated vs supported entity tokens.
   - **Side effects:** the KL divergence from the unchanged model on every answer token.
   - **Detector accuracy:** the span-max AUC of a logistic-regression detector at `L*`, fitted on the other fold.

   The steering strength used in stage 4 is the most negative `α` whose mean KL stays under `KL_MAX`.
4. **New answers.** `N_GEN` Hindi prompts that are not among the 60 are answered with the paper's settings under three conditions:
   unchanged, `ablate` and `steer−`. All three go through the same fp16 model, so the comparison is fair.
5. **Judge** every new answer with the same web-search judge as the Hindi annotations (repo prompt + Hindi note, gpt-oss-120b, Exa / DuckDuckGo).
6. **Results + figures.** Everything is read back from the saved files, so the plots can be remade later without a GPU.
   The hallucination rates come with paired 95% bootstrap intervals over prompts.

**Saving:** every result file goes to `/kaggle/working/steer-hindi/` and to `shreshtha2002/hallucination-probes-multiturn/steer-hindi/`.
The steering sweep is saved every `CKPT_EVERY` answers, generation after every batch, and the judge every `SAVE_EVERY` answers.
If the session stops, Run all again: every finished piece is read back (from the Hub if this machine lost it) and skipped.

**Kaggle:** GPU **T4 x2**, Internet **on**, Add-ons → Secrets → **`HF_TOKEN`**: the shreshtha2002 token (write + "Make calls to Inference Providers").
The account must have access to both models.

**Time / cost:** about 1 h per model on the GPUs, then 40–60 min of judging. 2 models × 3 conditions × 20 answers = 120 answers, about $6.

In [ ]:
import os, shutil, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "safetune", "rouge-score", "nltk", "absl-py", "ddgs", "openai",
                "pydantic>=2.6", "jaxtyping", "termcolor"], check=True)
if not os.path.isdir("repo"):     # the paper's code
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/obalcells/hallucination_probes", "repo"], check=True)
print("ready")

In [ ]:
# ---------------- settings ----------------
MODELS = {"llama": "meta-llama/Llama-3.1-8B-Instruct", "aya": "CohereLabs/aya-expanse-8b"}
ANNOTATED = {"llama": "shreshtha2002/longfact-hi-llama3.1-8b-instruct-q8-0-annotated",     # our 60 judge-annotated Hindi answers per model
             "aya": "shreshtha2002/longfact-hi-aya-expanse-8b-q8-0-annotated"}
PROMPT_SRC = "shreshthamodi02/longfact-hindi-paper"     # Hindi LongFact prompts (the paper's test split): new prompts come from here
HF_USER = "shreshtha2002"
OUT_REPO, RUN = f"{HF_USER}/hallucination-probes-multiturn", "steer-hindi"   # results: OUT_REPO/RUN/

# 1-3: direction, neurons, teacher-forced sweep (the probe notebooks' settings where they apply)
FOLDS, SEED, MAX_LEN = 2, 42, 2560
POS_WEIGHT = NEG_WEIGHT = 10.0            # used by label_example (the paper's token weights); not used for fitting here
LAYER_MIN_FRAC, LAYER_MAX_FRAC = 0.2, 0.8  # pick L* from the middle 60% of layers: early layers carry token identity,
                                          # the last 20% are excluded (SafeTune: min_layer_fraction 0.2, prune_layer_fraction 0.2)
TOP_K_NEURONS = 16                        # SafeTune identify_safety_neurons default
ALPHAS = [-8, -4, -2, -1, 1, 2, 4, 8]     # steering strengths, in units of the mean difference (CAA-style multipliers)
KL_MAX = 0.1                              # side-effect ceiling for the generation strength (SafeTune / Arditi et al. kl_threshold)
CKPT_EVERY = 5                            # answers between uploads (sweep, heads)
ATTN_WINDOW = 1024                        # heads stage: tokens per answer (cut from the left, as in the probe notebooks), eager attention
N_ATTN_EXAMPLES, TOP_HEADS = 3, 6         # attention patterns saved for this many answers, at the TOP_HEADS heads that write most along the direction
LOGIT_CHUNK = 128                         # positions per chunk when scoring the vocabulary (Aya: 256k vocab)
# 4: new answers (the paper's generation settings, App. C)
N_GEN, GEN_BATCH = 20, 10
TEMPERATURE, MAX_NEW_TOKENS = 0.1, 2048
CONDITIONS = ["base", "ablate", "steer-"]
# 5: judge (the same as the Hindi annotations)
JUDGE = "openai/gpt-oss-120b:groq"
JUDGE_TEMPERATURE, JUDGE_MAX_TOKENS, MAX_SEARCHES = 0.0, 8192, 15
EXA_MCP = "https://mcp.exa.ai/mcp?tools=web_search_exa"
ANN_WORKERS, SAVE_EVERY = 4, 5
HINDI_NOTE = True

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # less fragmentation (set before torch is imported)
OUT = f"{'/kaggle/working' if os.path.isdir('/kaggle/working') else 'out'}/{RUN}"
os.makedirs(f"{OUT}/figs", exist_ok=True)

In [ ]:
# ---------------- reused code: the paper's repo + ARENA 1.3.1 ----------------
import gc, html as htmllib, json, math, random, re, time
import numpy as np, pandas as pd, torch, torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from IPython.display import HTML, display
sys.path.insert(0, "repo")
from probe.loss import compute_probe_bce_loss, compute_probe_max_aggregation_loss   # paper's token BCE + span-max losses
from probe.dataset_converters import _MAP_LABEL_TO_SCALAR                          # NS / II -> 1, S -> 0
from utils.tokenization import find_string_in_tokens                                # paper's span -> token search
from utils.metrics import compute_clf_metrics                                       # paper's metrics
t = torch

# ---- ARENA 1.3.1 solutions, verbatim ----
def get_pca_components(activations, k=2):
    X = activations - activations.mean(dim=0)
    cov = X.t() @ X / (X.shape[0] - 1)
    eigenvalues, eigenvectors = t.linalg.eigh(cov)
    sorted_indices = t.argsort(eigenvalues, descending=True)
    return eigenvectors[:, sorted_indices[:k]]

class MMProbe(t.nn.Module):
    def __init__(self, direction, covariance=None, atol=1e-3, bias=None):
        super().__init__()
        self.direction = t.nn.Parameter(direction, requires_grad=False)
        self.bias = t.nn.Parameter(bias, requires_grad=False) if bias is not None else None
        self.inv = t.nn.Parameter(t.linalg.pinv(covariance, hermitian=True, atol=atol), requires_grad=False) if covariance is not None else None
    def forward(self, x, iid=False):
        if self.bias is not None:
            x = x - self.bias
        return t.sigmoid(x @ self.inv @ self.direction) if (iid and self.inv is not None) else t.sigmoid(x @ self.direction)
    @staticmethod
    def from_data(acts, labels, device="cpu", bias=True):
        acts, labels = acts.to(device), labels.to(device)
        pos_acts, neg_acts = acts[labels == 1], acts[labels == 0]
        pos_mean, neg_mean = pos_acts.mean(0), neg_acts.mean(0)
        centered = t.cat([pos_acts - pos_mean, neg_acts - neg_mean], dim=0)
        covariance = centered.t() @ centered / acts.shape[0]
        return MMProbe(pos_mean - neg_mean, covariance=covariance, bias=0.5 * (pos_mean + neg_mean) if bias else None).to(device)

class LRProbe(t.nn.Module):
    def __init__(self, d_in, scaler_mean=None, scaler_scale=None):
        super().__init__()
        self.net = t.nn.Sequential(t.nn.Linear(d_in, 1, bias=False), t.nn.Sigmoid())
        self.register_buffer("scaler_mean", scaler_mean); self.register_buffer("scaler_scale", scaler_scale)
    def forward(self, x):
        if self.scaler_mean is not None:
            x = (x - self.scaler_mean) / self.scaler_scale
        return self.net(x).squeeze(-1)
    @property
    def direction(self):
        return self.net[0].weight.data[0]
    @staticmethod
    def from_data(acts, labels, C=0.1, device="cpu"):
        X, y = acts.cpu().float().numpy(), labels.cpu().float().numpy()
        scaler = StandardScaler(); X_scaled = scaler.fit_transform(X)
        lr_model = LogisticRegression(C=C, random_state=42, fit_intercept=False, max_iter=1000).fit(X_scaled, y)
        probe = LRProbe(acts.shape[-1], t.tensor(scaler.mean_, dtype=t.float32), t.tensor(scaler.scale_, dtype=t.float32)).to(device)
        probe.net[0].weight.data[0] = t.tensor(lr_model.coef_[0], dtype=t.float32).to(device)
        return probe
log = lambda m: print(time.strftime("%H:%M:%S"), m, flush=True)

In [ ]:
# ---------------- Hub login, save / restore helpers, the shared code from the probe notebooks ----------------
import contextlib, threading, urllib.request
from concurrent.futures import ThreadPoolExecutor
from huggingface_hub import HfApi, hf_hub_download
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from safetune.interventions.steer.refusal_direction import RefusalDirectionModel          # SafeTune steer: ablate / add any direction
from safetune.instrumentation.interpret.safety_neurons import identify_safety_neurons, SafetyNeuronConfig   # SafeTune interpret
from safetune.config import ADVBENCH_REFUSAL_PREFIXES, HINDI_REFUSAL_PREFIXES          # SafeTune's refusal openings (English + Hindi)
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
assert HF_TOKEN, "No HF token: Kaggle > Add-ons > Secrets > add HF_TOKEN and tick it for this notebook"
api = HfApi(token=HF_TOKEN)
me = api.whoami()
assert HF_USER in [me["name"]] + [o["name"] for o in me.get("orgs", [])], f"HF_TOKEN belongs to '{me['name']}', not '{HF_USER}'"
api.create_repo(OUT_REPO, exist_ok=True)
log(f"HF login: {me['name']}; results -> https://huggingface.co/{OUT_REPO}/tree/main/{RUN}")

def fetch(path):
    """Local path of a saved result: this machine's copy, else the Hub copy, else None (not done yet)."""
    local = f"{OUT}/{path}"
    if not os.path.exists(local):
        try:
            src = hf_hub_download(OUT_REPO, f"{RUN}/{path}", token=HF_TOKEN)
            os.makedirs(os.path.dirname(local), exist_ok=True); shutil.copy(src, local)
        except Exception:
            return None
    return local

def push(path):
    """Upload one saved result (kept locally if the upload fails)."""
    try:
        api.upload_file(path_or_fileobj=f"{OUT}/{path}", path_in_repo=f"{RUN}/{path}", repo_id=OUT_REPO, commit_message=f"{RUN}: {path}")
    except Exception as ex:
        log(f"  upload of {path} failed (kept locally): {str(ex)[:100]}")

def read_jsonl(path):
    p = fetch(path)
    return [json.loads(l) for l in open(p, encoding="utf-8")] if p else []

def append_jsonl(path, rows):
    os.makedirs(os.path.dirname(f"{OUT}/{path}"), exist_ok=True)
    with open(f"{OUT}/{path}", "a", encoding="utf-8") as f:
        f.writelines(json.dumps(r, ensure_ascii=False) + "\n" for r in rows)

# ---------------- token labels: the paper's rules, applied to the LAST answer ----------------
def chat_ids(msgs, gen=False):
    return tok(tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=gen), add_special_tokens=False)["input_ids"]

def common_prefix(a, b):
    n = 0
    while n < min(len(a), len(b)) and a[n] == b[n]:
        n += 1
    return n

def label_example(row, max_len=MAX_LEN):
    conv = [{"role": m["role"], "content": m["content"]} for m in row["conversation"]]
    completion, full = conv[-1]["content"], chat_ids(conv)
    start = common_prefix(full, chat_ids(conv[:-1], gen=True))              # first token of the latest answer
    q0 = common_prefix(full, chat_ids(conv[:-2])) if len(conv) > 2 else 1   # first token of the latest user message
    ctx = min(64, max_len // 4)                                             # cut from the left; keep some context and BOS
    lo = max(0, len(full) - max_len)
    if lo > start - ctx:
        lo = max(0, start - ctx)
    ids = full[lo:lo + max_len]
    if lo > 0 and tok.bos_token_id is not None:
        ids = [tok.bos_token_id] + ids[1:]
    comp, ids_t = start - lo, torch.tensor(ids)
    labels = torch.full((len(ids),), -100.0); labels[comp:] = 0.0          # paper: answer tokens default 0, prompt ignored
    weights = torch.ones(len(ids))
    pos, neg, spans, cur = [], [], [], comp
    for a in sorted([a for a in row["annotations"] if a.get("index") is not None], key=lambda a: a["index"]):
        y = _MAP_LABEL_TO_SCALAR.get(a.get("label"), -100.0)
        if y == -100.0 or not a["span"] or a["span"] not in completion:
            continue
        try:                                                                # paper: search after the previous span
            s = find_string_in_tokens(a["span"], ids_t[cur:], tok)
        except (AssertionError, ValueError):
            continue
        a0, a1 = s.start + cur, s.stop + cur - 1
        if a0 < comp:
            continue
        cur = a0
        labels[a0:a1 + 1] = y; weights[a0:a1 + 1] = POS_WEIGHT if y == 1.0 else NEG_WEIGHT
        (pos if y == 1.0 else neg).append([a0, a1]); spans.append((a0, a1, y, a["span"], a.get("label")))
    for a0, a1 in pos:                                                      # positives win overlaps (paper)
        labels[a0:a1 + 1] = 1.0; weights[a0:a1 + 1] = POS_WEIGHT
    seg = {"earlier turns": (1, max(1, q0 - lo)), "current question": (max(1, q0 - lo), comp), "current answer": (comp, len(ids))}
    return {"ids": ids_t, "labels": labels, "weights": weights, "pos": pos, "neg": neg, "spans": spans, "comp": comp, "seg": seg, "row": row}

LR_CS = (1e-4, 1e-3, 1e-2, 1e-1, 1.0)           # L2 strengths tried for logistic regression (smaller C = stronger)
def pick_C(X, Y, G):
    # entity tokens of one answer are near-duplicates, so the strength is chosen by AUC on held-out ANSWERS (grouped CV inside the training fold)
    cv = GroupKFold(n_splits=min(3, len(set(G))))
    sc = {C: np.nanmean(cross_val_score(make_pipeline(StandardScaler(), LogisticRegression(C=C, fit_intercept=False, max_iter=1000)),
                                        X, Y, groups=G, cv=cv, scoring="roc_auc", error_score=np.nan, n_jobs=-1)) for C in LR_CS}
    sc = {C: v for C, v in sc.items() if not np.isnan(v)}
    return max(sc, key=sc.get) if sc else 0.1     # ARENA's default if no split had both classes


In [ ]:
# ---------------- data: the 60 annotated answers per model, and N_GEN new prompts (shared by both models) ----------------
ours = {}
for key, repo in ANNOTATED.items():
    rows = {}
    for l in open(hf_hub_download(repo, "progress/progress.jsonl", repo_type="dataset", token=HF_TOKEN), encoding="utf-8"):
        r = json.loads(l)
        if r.get("annotations") is not None:
            rows[r["prompt_id"]] = {"conversation": r["conversation"], "annotations": r["annotations"], "id": r["prompt_id"]}
    ours[key] = list(rows.values())
groups = sorted({r["id"] for rs in ours.values() for r in rs}); random.Random(SEED).shuffle(groups)
fold_of = {g: i % FOLDS for i, g in enumerate(groups)}            # the same 2 folds (by prompt) for both models
used = {next(m["content"] for m in r["conversation"] if m["role"] == "user").strip() for rs in ours.values() for r in rs}
src = [json.loads(l) for l in open(hf_hub_download(PROMPT_SRC, "data/test-longfact.jsonl", repo_type="dataset", token=HF_TOKEN), encoding="utf-8")]
new = [p for p in src if p["prompt_hi"].strip() not in used]
random.Random(SEED).shuffle(new)
gen_prompts = [{"prompt_id": f"gen-{i:03d}", "prompt": p["prompt_hi"], "prompt_en": p["prompt_en"], "subset": p["subset"]} for i, p in enumerate(new[:N_GEN])]
log(f"annotated answers: { {k: len(v) for k, v in ours.items()} }; {len(new)} unused Hindi prompts, {len(gen_prompts)} taken for new answers")
print(gen_prompts[0]["prompt"][:200])

In [ ]:
# ---------------- model loading, forward passes ----------------
def load(key):
    global tok, model, blocks, NL, D, MODEL
    MODEL = MODELS[key]
    tok = AutoTokenizer.from_pretrained(MODEL, token=HF_TOKEN)
    tok.padding_side = "left"                                   # for batched generation
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    NG = torch.cuda.device_count()     # T4 x2: keep ~5 GB free on GPU 0 (embeddings + output layer) and room on GPU 1 for activations
    kw = dict(device_map="auto", max_memory={i: ("9GiB" if i == 0 else "13GiB") for i in range(NG)} if NG > 1 else None,
              token=HF_TOKEN, attn_implementation="sdpa")
    try:
        model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16, **kw)     # T4 has no bf16: fp16
    except TypeError:
        model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, **kw)
    model.eval().requires_grad_(False)
    blocks = model.model.layers
    NL, D = len(blocks), model.config.hidden_size
    log(f"{MODEL}: {NL} layers, d_model {D}")

def unload():
    global model, blocks
    del model, blocks; gc.collect(); torch.cuda.empty_cache()
    if torch.cuda.is_available():
        log("GPU free after unloading: " + ", ".join(f"{torch.cuda.mem_get_info(i)[0] / 2**30:.1f} GiB" for i in range(torch.cuda.device_count())))

def forward(ids, layers, pos):
    """One pass: final hidden states [seq, D] and the residual stream (block output) at `pos` for each layer in `layers`."""
    got, hooks = {}, []
    for l in layers:
        hooks.append(blocks[l].register_forward_hook(lambda m, a, o, l=l: got.__setitem__(l, (o[0] if isinstance(o, tuple) else o)[0, pos.to((o[0] if isinstance(o, tuple) else o).device)].half().cpu())))
    try:
        with torch.no_grad():
            h = model.model(input_ids=ids[None].to(model.device)).last_hidden_state[0]
    finally:
        for hk in hooks:
            hk.remove()
    return h, got

def logprobs(h, ids, comp):
    """Next-token log-probabilities over the answer, in chunks (never the whole [seq, vocab] at once):
    returns, for each answer position i >= comp, the full log-softmax that predicted token i, chunk by chunk (a generator)."""
    W, scale = model.lm_head, getattr(model, "logit_scale", 1.0)      # Cohere multiplies logits by logit_scale; Llama has none
    for i in range(comp, len(ids), LOGIT_CHUNK):
        j = min(len(ids), i + LOGIT_CHUNK)
        yield i, j, (W(h[i - 1:j - 1].to(W.weight.device)).float() * scale).log_softmax(-1)

def entity_positions(e):
    """Positions of every judged entity token, and each span as indices into those positions."""
    pos = sorted({i for a0, a1, *_ in e["spans"] for i in range(a0, a1 + 1)})
    at = {p: k for k, p in enumerate(pos)}
    return torch.tensor(pos, dtype=torch.long), [([at[i] for i in range(a0, a1 + 1)], y) for a0, a1, y, *_ in e["spans"]]

def span_auc(scores, spans):
    """Span-max AUC (the paper's headline level): scores = per-position tensors, spans = per-answer [(idx, y)]."""
    p = np.array([float(s[idx].max()) for s, sp in zip(scores, spans) for idx, _ in sp])
    y = np.array([y for sp in spans for _, y in sp])
    return compute_clf_metrics((p > np.median(p)).astype(float), y, p)["auc"]

In [ ]:
# ---------------- stages 1-4 for one model (each stage skips itself if its results are already saved) ----------------
def stage_direction(key):
    """1. difference of means (hallucinated − supported entity tokens) at every layer, cross-fitted; pick L*."""
    if fetch(f"{key}/direction.pt"):
        return torch.load(f"{OUT}/{key}/direction.pt", weights_only=False)
    ex = []
    for r in ours[key]:
        e = label_example(r)
        pos, spans = entity_positions(e)
        if len(pos):
            _, got = forward(e["ids"], range(NL), pos)
            ex.append({"id": r["id"], "fold": fold_of[r["id"]], "acts": torch.stack([got[l] for l in range(NL)], 1),   # [ntok, NL, D]
                       "y": e["labels"][pos], "spans": spans})
    def mean_diff(xs):                                       # [NL, D]: mean(hallucinated) − mean(supported), over entity tokens
        s, n = {1: 0.0, 0: 0.0}, {1: 0, 0: 0}                # running sums: never holds all tokens in float32
        for x in xs:
            for y in (1, 0):
                s[y] = s[y] + x["acts"][x["y"] == y].float().sum(0); n[y] += int((x["y"] == y).sum())
        return s[1] / n[1] - s[0] / n[0]
    auc = np.zeros((FOLDS, NL)); dirs = []
    for f in range(FOLDS):
        md_ = mean_diff([x for x in ex if x["fold"] != f]); dirs.append(md_)
        test = [x for x in ex if x["fold"] == f]
        for l in range(NL):
            auc[f, l] = span_auc([x["acts"][:, l].float() @ md_[l] for x in test], [x["spans"] for x in test])
    lo, hi = int(NL * LAYER_MIN_FRAC), int(NL * LAYER_MAX_FRAC)
    L = lo + int(np.argmax(auc.mean(0)[lo:hi]))
    out = {"L": L, "auc": auc, "fold_dirs": [d[L] for d in dirs], "all": mean_diff(ex),        # all = every layer, all 60 answers
           "test_ids": [[x["id"] for x in ex if x["fold"] == f] for f in range(FOLDS)],
           "ents_L": [{"id": x["id"], "fold": x["fold"], "acts": x["acts"][:, L].clone(), "y": x["y"], "spans": x["spans"]} for x in ex]}
    os.makedirs(f"{OUT}/{key}", exist_ok=True); torch.save(out, f"{OUT}/{key}/direction.pt"); push(f"{key}/direction.pt")
    log(f"[{key}] direction: L* = {L} (held-out span-max AUC {auc.mean(0)[L]:.3f}; best of layers {lo}-{hi - 1})")
    del ex; gc.collect()
    return out

def stage_neurons(key, dr):
    """2. SafeTune interpret: MLP neurons whose down_proj column writes along the direction (weight mode: no forward passes)."""
    if fetch(f"{key}/neurons.json"):
        return
    rep = identify_safety_neurons(model, {l: dr["all"][l] for l in range(NL)}, SafetyNeuronConfig(method="weight", top_k_per_layer=TOP_K_NEURONS))
    json.dump({"direction_layer": dr["L"], "method": rep.method, "target_module": rep.target_module,
               "per_layer": {str(l): v for l, v in rep.per_layer.items()}}, open(f"{OUT}/{key}/neurons.json", "w"))
    push(f"{key}/neurons.json"); log(f"[{key}] neurons: top {TOP_K_NEURONS} per layer saved")

STATS = ["to_question", "to_answer", "to_first_token", "entropy"]
def set_attn(impl):
    try:
        model.set_attn_implementation(impl)
    except Exception:
        model.config._attn_implementation = impl

def stage_heads(key, dr):
    """2b. per head and per judged entity: write along the direction (weights and activations) and where it attends; saved every CKPT_EVERY answers."""
    path = f"{key}/heads.pt"
    H = torch.load(fetch(path), weights_only=False) if fetch(path) else None
    if H is not None and H["complete"]:
        return H
    L = dr["L"]; d = dr["all"][L].float(); d = d / d.norm()
    NH = model.config.num_attention_heads; HD = getattr(model.config, "head_dim", None) or D // NH
    Wo = [blocks[l].self_attn.o_proj.weight for l in range(NL)]                                  # [D, NH * HD]
    u = [(W.float().T @ d.to(W.device)).view(NH, HD).cpu() for W in Wo]                          # head h writes z_h . u[l][h] along d
    align = torch.stack([u[l].norm(dim=1) / Wo[l].float().view(D, NH, HD).norm(dim=(0, 2)).cpu() for l in range(NL)])   # [NL, NH] weight alignment
    if H is None:
        H = {"L": L, "NH": NH, "stats": STATS, "weight_align": align, "spans": [], "done": [], "examples": [], "complete": False}
    set_attn("eager")                                                                            # attention weights need eager attention
    try:
        for n_new, r in enumerate([r for r in ours[key] if r["id"] not in H["done"]], 1):
            e = label_example(r, max_len=ATTN_WINDOW); pos, spans = entity_positions(e)
            if len(pos):
                q0, comp = e["seg"]["current question"][0], e["comp"]
                got, hooks = {}, []
                for l in range(NL):
                    hooks.append(blocks[l].self_attn.o_proj.register_forward_pre_hook(lambda m, a, l=l: got.__setitem__(("z", l),
                        (a[0][0, pos.to(a[0].device)].float().view(len(pos), NH, HD) * u[l].to(a[0].device)).sum(-1).cpu())))       # [n, NH]
                    hooks.append(blocks[l].self_attn.register_forward_hook(lambda m, a, o, l=l: got.__setitem__(("a", l), (lambda A: torch.stack([
                        A[..., q0:comp].sum(-1), A[..., comp:].sum(-1), A[..., 0], -(A * (A + 1e-9).log()).sum(-1)], -1).transpose(0, 1).cpu())(
                        o[1][0][:, pos.to(o[1].device)].float()))))                                                                    # [n, NH, 4]
                hooks.append(blocks[L].register_forward_hook(lambda m, a, o: got.__setitem__("proj", (lambda h: (h[0, pos.to(h.device)].float() @ d.to(h.device)).cpu())(
                    o[0] if isinstance(o, tuple) else o))))                                                                            # [n] projection on d at L*
                try:
                    with torch.no_grad():
                        model.model(input_ids=e["ids"][None].to(model.device))
                finally:
                    for hk in hooks:
                        hk.remove()
                Z = torch.stack([got["z", l] for l in range(NL)], 1)                                   # [n, NL, NH]
                Aa = torch.stack([got["a", l] for l in range(NL)], 1)                                  # [n, NL, NH, 4]
                for idx, y in spans:
                    H["spans"].append({"id": r["id"], "fold": fold_of[r["id"]], "y": y, "proj": float(got["proj"][idx].max()),
                                       "contrib": Z[idx].mean(0).half(), "attn": Aa[idx].mean(0).half()})
                del got, Z, Aa
            H["done"].append(r["id"])
            if n_new % CKPT_EVERY == 0:
                torch.save(H, f"{OUT}/{path}"); push(path); log(f"[{key}] heads: {len(H['done'])} / {len(ours[key])} answers")
        # attention patterns of the TOP_HEADS heads (by how differently they write along d for hallucinated vs supported), around a hallucinated entity
        y = np.array([sp["y"] for sp in H["spans"]]); C = torch.stack([sp["contrib"].float() for sp in H["spans"]]).numpy()
        gap = C[y == 1].mean(0) - C[y == 0].mean(0)
        top = [(int(i // NH), int(i % NH)) for i in np.argsort(-np.abs(gap).ravel())[:TOP_HEADS]]
        H["top_heads"], H["examples"] = top, []
        for r in [r for r in ours[key] if any(sp["id"] == r["id"] and sp["y"] == 1 for sp in H["spans"])][:N_ATTN_EXAMPLES]:
            e = label_example(r, max_len=ATTN_WINDOW); a0, a1 = e["pos"][0]; w = slice(max(0, a0 - 48), min(len(e["ids"]), a1 + 16))
            got, hooks = {}, [blocks[l].self_attn.register_forward_hook(lambda m, a, o, l=l: got.__setitem__(l, o[1][0][:, w, w].float().cpu())) for l in {l for l, _ in top}]
            try:
                with torch.no_grad():
                    model.model(input_ids=e["ids"][None].to(model.device))
            finally:
                for hk in hooks:
                    hk.remove()
            H["examples"].append({"id": r["id"], "tokens": [tok.decode([i]) for i in e["ids"][w]], "span": (a0 - w.start, a1 - w.start),
                                  "patterns": {f"L{l}H{h}": got[l][h].half() for l, h in top}})
        H["complete"] = True
    finally:
        set_attn("sdpa")
    torch.save(H, f"{OUT}/{path}"); push(path); log(f"[{key}] heads done: {len(H['spans'])} entities; top heads {H['top_heads']}")
    return H

def intervention(d, cond, strength, L):
    """SafeTune RefusalDirectionModel with our direction: 'ablate' at every layer, or 'steer' (+ strength · unit d) at L."""
    if cond == "ablate":
        return RefusalDirectionModel(model, d, mode="ablate")
    return RefusalDirectionModel(model, d, mode="steer", strength=strength, layers=[L])

def stage_sweep(key, dr):
    """3. teacher-forced: remove / steer the held-out direction on each held-out answer; probability shift, KL, detector scores."""
    path = f"{key}/sweep.jsonl"
    done = {(r["id"], r["cond"]) for r in read_jsonl(path)}
    L = dr["L"]
    conds = [("ablate", 0.0)] + [(f"steer{a:+g}", float(a)) for a in ALPHAS]
    n_new = 0
    for f in range(FOLDS):
        d, gap = dr["fold_dirs"][f].float(), float(dr["fold_dirs"][f].float().norm())          # steer by a · (mean difference)
        tr = [x for x in dr["ents_L"] if x["fold"] != f]                                           # detector: logistic regression at L*,
        X, Y = torch.cat([x["acts"] for x in tr]).float(), torch.cat([x["y"] for x in tr])         # fitted on the other fold (C picked by
        G = np.concatenate([[i] * len(x["y"]) for i, x in enumerate(tr)])                          # grouped CV, as in the probe notebooks)
        det = LRProbe.from_data(X, Y, C=pick_C(X.numpy(), Y.numpy(), G))
        for r in [r for r in ours[key] if r["id"] in dr["test_ids"][f]]:
            todo = [(c, a) for c, a in conds if (r["id"], c) not in done]
            if not todo:
                continue
            e = label_example(r); pos, spans = entity_positions(e)
            if not len(pos):
                continue
            ids, comp = e["ids"], e["comp"]
            hb, gb = forward(ids, [L], pos)
            rows = [{"id": r["id"], "fold": f, "cond": "base", "alpha": 0.0, "kl": 0.0, "dlogp_ns": 0.0, "dlogp_s": 0.0, "dlogp_other": 0.0,
                     "det_span_scores": [float(det(gb[L].float())[idx].max()) for idx, _ in spans], "span_y": [y for _, y in spans]}] \
                   if (r["id"], "base") not in done else []
            lab = e["labels"]; in_span = torch.zeros(len(ids), dtype=torch.bool); in_span[pos] = True
            for c, a in todo:
                with intervention(d, c, a * gap, L):
                    hc, gc_ = forward(ids, [L], pos)
                kl, dl = torch.zeros(len(ids)), torch.zeros(len(ids))
                with torch.no_grad():
                    for (i, j, lb), (_, _, lc) in zip(logprobs(hb, ids, comp), logprobs(hc, ids, comp)):
                        kl[i:j] = (lb.exp() * (lb - lc)).sum(-1).cpu()
                        tgt = ids[i:j].to(lb.device)[:, None]
                        dl[i:j] = (lc.gather(1, tgt) - lb.gather(1, tgt))[:, 0].cpu()
                ans = torch.arange(len(ids)) >= comp
                rows.append({"id": r["id"], "fold": f, "cond": c, "alpha": a, "kl": float(kl[ans].mean()),
                             "dlogp_ns": float(dl[in_span & (lab == 1)].mean()), "dlogp_s": float(dl[in_span & (lab == 0)].mean()),
                             "dlogp_other": float(dl[ans & ~in_span].mean()),
                             "det_span_scores": [float(det(gc_[L].float())[idx].max()) for idx, _ in spans], "span_y": [y for _, y in spans]})
                del hc
            append_jsonl(path, rows); n_new += 1
            if n_new % CKPT_EVERY == 0:
                push(path); log(f"[{key}] sweep: {len({x[0] for x in done}) + n_new} / {len(ours[key])} answers")
            del hb
    if n_new:
        push(path)
    sw = pd.DataFrame(read_jsonl(path))
    kl = sw[sw.cond.str.startswith("steer")].groupby("alpha").kl.mean()
    ok = [a for a in sorted(ALPHAS) if a < 0 and kl.get(float(a), np.inf) <= KL_MAX]
    alpha = ok[0] if ok else -1.0
    log(f"[{key}] sweep done; mean KL by alpha: {kl.round(3).to_dict()} -> steer- uses alpha = {alpha}" + ("" if ok else f" (none under KL {KL_MAX})"))
    return alpha

def stage_generate(key, dr, alpha):
    """4. new Hindi answers under each condition (direction and L* from all 60 answers), saved after every batch."""
    L, d = dr["L"], dr["all"][dr["L"]].float()
    for cond in CONDITIONS:
        path = f"{key}/gen_{cond}.jsonl"
        have = {r["prompt_id"] for r in read_jsonl(path)}
        todo = [p for p in gen_prompts if p["prompt_id"] not in have]
        for b in range(0, len(todo), GEN_BATCH):
            ps = todo[b:b + GEN_BATCH]
            enc = tok([tok.apply_chat_template([{"role": "user", "content": p["prompt"]}], tokenize=False, add_generation_prompt=True) for p in ps],
                      return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
            torch.manual_seed(SEED + b)
            ctx = intervention(d, cond, alpha * float(d.norm()), L) if cond != "base" else contextlib.nullcontext()
            tic = time.time()
            with ctx, torch.no_grad():
                out = model.generate(**enc, do_sample=True, temperature=TEMPERATURE, top_p=1.0, max_new_tokens=MAX_NEW_TOKENS, pad_token_id=tok.pad_token_id)
            texts = tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
            append_jsonl(path, [{"model": MODEL, "key": key, "cond": cond, "alpha": alpha if cond == "steer-" else None, "layer": L,
                                 "prompt_id": p["prompt_id"], "subset": p["subset"], "lang": "hi",
                                 "conversation": [{"role": "user", "content": p["prompt"]}, {"role": "assistant", "content": tx.strip()}]}
                                for p, tx in zip(ps, texts)])
            push(path); del out, enc; torch.cuda.empty_cache()
            log(f"[{key}] {cond}: {len(have) + b + len(ps)} / {len(gen_prompts)} answers ({time.time() - tic:.0f}s)")

In [ ]:
# ---------------- run stages 1-4: Llama, then Aya (one model on the GPUs at a time) ----------------
for key in MODELS:
    gens_done = all(len(read_jsonl(f"{key}/gen_{c}.jsonl")) >= len(gen_prompts) for c in CONDITIONS)
    heads_done = fetch(f"{key}/heads.pt") and torch.load(f"{OUT}/{key}/heads.pt", weights_only=False)["complete"]
    if gens_done and heads_done and fetch(f"{key}/neurons.json") and fetch(f"{key}/sweep.jsonl"):
        log(f"[{key}] stages 1-4 already done"); continue
    load(key)
    dr = stage_direction(key)
    stage_neurons(key, dr)
    stage_heads(key, dr)
    alpha = stage_sweep(key, dr)
    stage_generate(key, dr, alpha)
    unload()

In [ ]:
# ---------------- stage 5: judge every new answer (the repo's prompt and code + the Hindi note, as for the Hindi annotations) ----------------
import ast, importlib.util, logging
from typing import List
from ddgs import DDGS
from openai import OpenAI
from rouge_score import rouge_scorer
from utils.parsing import parse_and_validate_json
import utils.string_utils as string_utils
from utils.string_utils import try_matching_span_in_text

spec = importlib.util.spec_from_file_location("data_models", "repo/annotation_pipeline/data_models.py")
data_models = importlib.util.module_from_spec(spec); spec.loader.exec_module(data_models)
AnnotatedSpan = data_models.AnnotatedSpan
# format_prompt and assign_span_positions, taken verbatim from annotate.py (importing the file itself would need Anthropic's safety-tooling)
repo_fns = {"List": List, "AnnotatedSpan": AnnotatedSpan, "try_matching_span_in_text": try_matching_span_in_text,
            "logger": logging.getLogger("annotate")}
for node in ast.parse(open("repo/annotation_pipeline/annotate.py", encoding="utf-8").read()).body:
    if isinstance(node, ast.FunctionDef) and node.name in ("format_prompt", "assign_span_positions"):
        exec(compile(ast.Module([node], []), "annotate.py", "exec"), repo_fns)
format_prompt, assign_span_positions = repo_fns["format_prompt"], repo_fns["assign_span_positions"]
PROMPT = open("repo/annotation_pipeline/entity_annotation.prompt", encoding="utf-8").read().strip()   # as in annotate.py

if HINDI_NOTE:                 # ROUGE's default tokenizer deletes non-Latin text (as in the Hindi annotation notebooks)
    class UnicodeTokenizer:
        def tokenize(self, text):
            return [w for w in re.split(r"[\s.,;:!?\"'()\[\]\-‐‑‒–—―।]+", text.lower()) if w]
    string_utils.ROUGE_SCORER = rouge_scorer.RougeScorer(["rougeL"], tokenizer=UnicodeTokenizer())

client = OpenAI(base_url="https://router.huggingface.co/v1", api_key=HF_TOKEN, max_retries=6, timeout=600)

def exa_mcp(query):
    # minimal MCP (streamable HTTP) client: initialize -> initialized -> tools/call web_search_exa
    H = {"Content-Type": "application/json", "Accept": "application/json, text/event-stream",
         "User-Agent": "Mozilla/5.0 (hallucination-annotator)"}
    def rpc(body, sid=None):
        req = urllib.request.Request(EXA_MCP, data=json.dumps(body).encode(), headers={**H, **({"Mcp-Session-Id": sid} if sid else {})})
        with urllib.request.urlopen(req, timeout=60) as r:
            sid = r.headers.get("Mcp-Session-Id") or sid
            txt = r.read().decode("utf-8")
        data = [l[5:].strip() for l in txt.splitlines() if l.startswith("data:")]
        msg = json.loads(data[-1]) if data else (json.loads(txt) if txt.strip() else {})
        if "error" in msg:
            raise RuntimeError(msg["error"])
        return sid, msg
    sid, _ = rpc({"jsonrpc": "2.0", "id": 1, "method": "initialize",
                  "params": {"protocolVersion": "2025-03-26", "capabilities": {}, "clientInfo": {"name": "annotator", "version": "0.1"}}})
    rpc({"jsonrpc": "2.0", "method": "notifications/initialized"}, sid)
    _, msg = rpc({"jsonrpc": "2.0", "id": 2, "method": "tools/call",
                  "params": {"name": "web_search_exa", "arguments": {"query": query, "numResults": 5}}}, sid)
    return "\n\n".join(c.get("text", "") for c in msg["result"]["content"])

EXA_COOLDOWN = {"until": 0}
def web_search(query):            # Exa MCP when it isn't rate-limiting us; otherwise DuckDuckGo
    if time.time() > EXA_COOLDOWN["until"]:
        try:
            return exa_mcp(query)[:2500]          # keeps the judge's context (and your credits) small
        except Exception as e:
            EXA_COOLDOWN["until"] = time.time() + 600
            log(f"  Exa MCP failed ({str(e)[:80]}); DuckDuckGo only for the next 10 min")
    for attempt in range(3):
        try:
            hits = DDGS().text(query, max_results=5)
            return "\n\n".join(f"{h.get('title')}\n{h.get('href')}\n{h.get('body')}" for h in hits) or "No results."
        except Exception:
            time.sleep(3 * 2 ** attempt)
    log(f"  SEARCH FAILED: {query[:60]}")
    return "Search failed."

SEARCH = {"name": "web_search", "description": "Search the web. Returns the top results (title, URL, snippet).",
          "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}}

def judge(user_prompt):
    """The repo's prompt as one user message; the judge may call web_search up to MAX_SEARCHES times."""
    n, found = 0, []
    tools = [{"type": "function", "function": SEARCH}]
    msgs = [{"role": "user", "content": user_prompt}]
    for _ in range(MAX_SEARCHES + 5):            # tools stay on (Groq rejects a tool call when tools are removed)
        m = client.chat.completions.create(model=JUDGE, messages=msgs, tools=tools, temperature=JUDGE_TEMPERATURE,
                                           max_tokens=JUDGE_MAX_TOKENS).choices[0].message
        if not m.tool_calls:
            return m.content or ""
        msgs.append({"role": "assistant", "content": m.content or "", "tool_calls": [tc.model_dump() for tc in m.tool_calls]})
        for tc in m.tool_calls:
            n += 1
            query = json.loads(tc.function.arguments or "{}").get("query", "")
            result = web_search(query) if n <= MAX_SEARCHES else "Search limit reached. Give your final answer now."
            if n <= MAX_SEARCHES:
                found.append(f"[search: {query}]\n{result}")
            msgs.append({"role": "tool", "tool_call_id": tc.id, "content": result})
    # rare: it kept searching past the cap -> one last call without tools, with the results it already gathered
    final = (user_prompt + "\n\nWeb search results already gathered:\n\n" + "\n\n".join(found)
             + "\n\nNo more searches are possible. Return the JSON array now.")
    return client.chat.completions.create(model=JUDGE, messages=[{"role": "user", "content": final}],
                                          temperature=JUDGE_TEMPERATURE, max_tokens=JUDGE_MAX_TOKENS).choices[0].message.content or ""

HINDI = ("\n\nNote: the instruction and the completion are in Hindi. Copy every \"span\" exactly as it appears in the completion, "
         "in Devanagari script, without translating or transliterating it. You may search in English or Hindi. "
         "Write the verification_note in English.")
ANNOTATOR = f"hf/{JUDGE}+exa-mcp/ddgs" + ("+hindi-note" if HINDI_NOTE else "")

def annotate(row):
    """Same steps as the repo's annotate_completion (instruction = conversation[-2], completion = conversation[-1])."""
    instruction, completion = row["conversation"][-2]["content"], row["conversation"][-1]["content"]
    prompt = format_prompt(instruction, completion, PROMPT) + (HINDI if HINDI_NOTE else "")
    for attempt in range(3):
        try:
            spans = parse_and_validate_json(judge(prompt), List[AnnotatedSpan], allow_partial=True)
            spans = assign_span_positions(spans, completion)
            log(f"  done {row['prompt_id']}: {len(spans)} entities")
            return {**row, "annotations": [s.model_dump() for s in spans], "annotator": ANNOTATOR}
        except Exception as e:
            log(f"  attempt {attempt + 1} failed for {row['prompt_id']}: {str(e)[:300]}")
    return {**row, "annotations": None, "annotator": None}   # failed; left unlabelled

ANN = "annotations.jsonl"
saved = {f"{r['key']}|{r['cond']}|{r['prompt_id']}": r for r in read_jsonl(ANN)}
answers = [r for key in MODELS for c in CONDITIONS for r in read_jsonl(f"{key}/gen_{c}.jsonl")]
lock, since_upload = threading.Lock(), [0]

def annotate_and_save(row):
    k = f"{row['key']}|{row['cond']}|{row['prompt_id']}"
    if k in saved:
        return saved[k]
    r = annotate(row)
    if r["annotations"] is not None:               # failures are not saved, so they are retried on the next run
        with lock:
            saved[k] = r; append_jsonl(ANN, [r]); since_upload[0] += 1
            if since_upload[0] >= SAVE_EVERY:
                since_upload[0] = 0; push(ANN)
    return r

log(f"stage 5: {len(answers)} answers, {len(saved)} already judged")
with ThreadPoolExecutor(ANN_WORKERS) as pool:
    judged = list(pool.map(annotate_and_save, answers))
push(ANN)
log(f"stage 5 done: {sum(r['annotations'] is not None for r in judged)} judged, {sum(r['annotations'] is None for r in judged)} failed (run this cell again to retry them)")

In [ ]:
# ---------------- stage 6: results + figures, from the saved files only (no GPU needed; rerun any time) ----------------
def savefig(name):
    plt.tight_layout(); plt.savefig(f"{OUT}/figs/{name}.png", dpi=300); plt.show()
NAME = {"llama": "Llama-3.1-8B", "aya": "Aya Expanse 8B"}
COL = {"base": "#8a8f96", "ablate": "#2e6fb8", "steer-": "#2e9b45"}
drs = {k: torch.load(fetch(f"{k}/direction.pt"), weights_only=False) for k in MODELS if fetch(f"{k}/direction.pt")}

# 1. where the direction separates hallucinated from supported entities (held-out, per layer)
fig, ax = plt.subplots(figsize=(8, 3.8))
for (k, dr), c in zip(drs.items(), ["#d33a2c", "#6f8fb8"]):
    x = np.arange(dr["auc"].shape[1]) / (dr["auc"].shape[1] - 1)
    ax.plot(x, dr["auc"].mean(0), "-o", ms=3, color=c, label=f"{NAME[k]} (L* = {dr['L']})")
    ax.axvline(dr["L"] / (dr["auc"].shape[1] - 1), color=c, ls="--", lw=1)
ax.axhline(0.5, ls=":", color="k", lw=1); ax.set_xlabel("relative depth"); ax.set_ylabel("span-max AUC (held-out)"); ax.legend(fontsize=8)
ax.set_title("hallucination direction (difference of means) by layer"); savefig("1_direction_by_layer")

# 2. SafeTune interpret: how strongly the top MLP neurons write along the direction, by layer
fig, axs = plt.subplots(1, len(drs), figsize=(5.5 * len(drs), 3.8), squeeze=False)
for ax, k in zip(axs[0], drs):
    nj = json.load(open(fetch(f"{k}/neurons.json")))
    M = np.array([[s for _, s in nj["per_layer"].get(str(l), [])] + [np.nan] * (TOP_K_NEURONS - len(nj["per_layer"].get(str(l), []))) for l in range(drs[k]["auc"].shape[1])])
    im = ax.imshow(M.T, aspect="auto", cmap="viridis", origin="lower"); plt.colorbar(im, ax=ax, label="|cos(down_proj column, direction)|")
    ax.set_xlabel("layer"); ax.set_ylabel("neuron rank"); ax.set_title(f"{NAME[k]}: top {TOP_K_NEURONS} neurons per layer")
savefig("2_neurons")

# 3. teacher-forced sweep: does moving along the direction change confidence in hallucinated vs supported entities?
sw = {k: pd.DataFrame(read_jsonl(f"{k}/sweep.jsonl")) for k in drs}
sw = {k: v for k, v in sw.items() if len(v)}
fig, axs = plt.subplots(1, 3, figsize=(14, 3.8))
for (k, s), c in zip(sw.items(), ["#d33a2c", "#6f8fb8"]):
    st = s[s.cond.str.startswith("steer")].groupby("alpha")[["dlogp_ns", "dlogp_s", "kl"]].mean()
    ab = s[s.cond == "ablate"][["dlogp_ns", "dlogp_s", "kl"]].mean()
    axs[0].plot(st.index, st.dlogp_ns, "-o", color=c, label=f"{NAME[k]}: hallucinated"); axs[0].plot(st.index, st.dlogp_s, "--o", color=c, mfc="white", label=f"{NAME[k]}: supported")
    axs[1].plot(st.index, st.dlogp_ns - st.dlogp_s, "-o", color=c, label=NAME[k]); axs[1].axhline(ab.dlogp_ns - ab.dlogp_s, color=c, ls=":", label=f"{NAME[k]}: ablate")
    axs[2].plot(st.index, st.kl, "-o", color=c, label=NAME[k]); axs[2].axhline(ab.kl, color=c, ls=":", label=f"{NAME[k]}: ablate")
axs[0].set_ylabel("Δ log p of entity tokens"); axs[1].set_ylabel("Δ log p: hallucinated − supported"); axs[2].set_ylabel("KL from unchanged (nats / token)")
axs[2].axhline(KL_MAX, color="k", ls="--", lw=1); axs[2].set_yscale("log")
for ax in axs:
    ax.axhline(0, color="k", lw=0.6); ax.set_xlabel("steering strength α (× mean difference)"); ax.legend(fontsize=6)
savefig("3_sweep")

# 4. does a detector still find hallucinations once the direction is removed? (held-out span-max AUC at L*)
det = []
for k, s in sw.items():
    for c in ["base", "ablate"] + [f"steer{a:+g}" for a in ALPHAS]:
        r = s[s.cond == c]
        if len(r):
            p, y = np.concatenate(r.det_span_scores.values), np.concatenate(r.span_y.values)
            det.append({"model": NAME[k], "cond": c, "auc": compute_clf_metrics((p > 0.5).astype(float), y, p)["auc"]})
det = pd.DataFrame(det); det.to_csv(f"{OUT}/detector_auc.csv", index=False)
if len(det):
    det[det.cond.isin(["base", "ablate"])].pivot(index="model", columns="cond", values="auc")[["base", "ablate"]].plot.bar(rot=0, figsize=(5.5, 3.5), color=[COL["base"], COL["ablate"]])
    plt.axhline(0.5, ls=":", color="k"); plt.ylim(0, 1); plt.ylabel("detector span-max AUC at L*"); plt.title("logistic-regression detector, direction removed"); savefig("4_detector")

# 5. behaviour: the judge on the new answers, paired over prompts, with 95% bootstrap intervals
ann = pd.DataFrame(read_jsonl("annotations.jsonl"))
PREF = tuple(p.lower() for p in ADVBENCH_REFUSAL_PREFIXES + HINDI_REFUSAL_PREFIXES)
def per_answer(r):
    labs = [a["label"] for a in r["annotations"] if a["label"] and a["index"] is not None]
    text = r["conversation"][-1]["content"]; letters = [ch for ch in text if ch.isalpha()]
    return {"model": NAME[r["key"]], "cond": r["cond"], "prompt_id": r["prompt_id"], "entities": len(labs),
            "ns": labs.count("Not Supported"), "ii": labs.count("Insufficient Information"),
            "abstain": float(text.strip().lower().startswith(PREF)), "devanagari": sum("\u0900" <= ch <= "\u097f" for ch in letters) / max(1, len(letters)),
            "chars": len(text)}
pa = pd.DataFrame([per_answer(r) for _, r in ann.iterrows()]) if len(ann) else pd.DataFrame()
if len(pa):
    pa.to_csv(f"{OUT}/per_answer.csv", index=False)
    rng = np.random.default_rng(SEED)
    def rate(d):                                            # entity-level hallucination rate (NS + II, the paper's positive class)
        return 100 * (d.ns.sum() + d.ii.sum()) / max(1, d.entities.sum())
    rows = []
    for m, dm in pa.groupby("model"):
        pids = sorted(set.intersection(*(set(dm[dm.cond == c].prompt_id) for c in CONDITIONS)))       # prompts judged under every condition
        dm = dm[dm.prompt_id.isin(pids)].set_index("prompt_id")
        boots = [rng.choice(pids, len(pids)) for _ in range(2000)]
        for c in CONDITIONS:
            dc, db = dm[dm.cond == c], dm[dm.cond == "base"]
            diff = [rate(dc.loc[b]) - rate(db.loc[b]) for b in boots]
            rows.append({"model": m, "cond": c, "prompts": len(pids), "halluc_%": rate(dc),
                         "ci_lo": np.percentile([rate(dc.loc[b]) for b in boots], 2.5), "ci_hi": np.percentile([rate(dc.loc[b]) for b in boots], 97.5),
                         "vs_base_pp": rate(dc) - rate(db), "vs_base_lo": np.percentile(diff, 2.5), "vs_base_hi": np.percentile(diff, 97.5),
                         "NS_%": 100 * dc.ns.sum() / max(1, dc.entities.sum()), "entities_per_answer": dc.entities.mean(),
                         "abstain_%": 100 * dc.abstain.mean(), "devanagari_%": 100 * dc.devanagari.mean(), "chars": dc.chars.mean()})
    res = pd.DataFrame(rows); res.to_csv(f"{OUT}/behaviour.csv", index=False)
    display(res.round(2))
    fig, axs = plt.subplots(1, 4, figsize=(15, 3.6)); ms = list(res.model.unique()); w = 0.8 / len(CONDITIONS)
    for j, c in enumerate(CONDITIONS):
        r = res[res.cond == c].set_index("model").loc[ms]; x = np.arange(len(ms)) + j * w
        axs[0].bar(x, r["halluc_%"], w, color=COL[c], label=c, yerr=[r["halluc_%"] - r.ci_lo, r.ci_hi - r["halluc_%"]], capsize=3)
        axs[1].bar(x, r.entities_per_answer, w, color=COL[c]); axs[2].bar(x, r["abstain_%"], w, color=COL[c]); axs[3].bar(x, r["devanagari_%"], w, color=COL[c])
    for ax, t in zip(axs, ["hallucinated entities % (NS + II), 95% CI", "entities per answer", "answers opening with a refusal %", "Devanagari letters %"]):
        ax.set_xticks(np.arange(len(ms)) + w); ax.set_xticklabels(ms, fontsize=8); ax.set_title(t, fontsize=9)
    for ax in (axs[0], axs[2], axs[3]):
        ax.set_ylim(0, 100)
    axs[0].legend(fontsize=7); savefig("5_behaviour")

# 6. heads: per-head tables (saved), heatmaps, correlation matrices and attention patterns
from sklearn.metrics import roc_auc_score
hs = {k: torch.load(fetch(f"{k}/heads.pt"), weights_only=False) for k in MODELS if fetch(f"{k}/heads.pt")}
tabs = []
for k, H in hs.items():
    y = np.array([sp["y"] for sp in H["spans"]])
    Cn = torch.stack([sp["contrib"].float() for sp in H["spans"]]).numpy()                  # [spans, NL, NH]
    An = torch.stack([sp["attn"].float() for sp in H["spans"]]).numpy()                     # [spans, NL, NH, 4]
    auc = lambda X: np.apply_along_axis(lambda v: roc_auc_score(y, v), 0, X)               # per-head AUC: hallucinated vs supported entities
    t = {"weight_align": H["weight_align"].numpy(), "write_auc": auc(Cn), "write_gap": Cn[y == 1].mean(0) - Cn[y == 0].mean(0)}
    t.update({f"{st}_auc": auc(An[..., i]) for i, st in enumerate(H["stats"])})
    NLk, NHk = t["write_auc"].shape
    tabs.append(pd.DataFrame({"model": NAME[k], "layer": np.repeat(np.arange(NLk), NHk), "head": np.tile(np.arange(NHk), NLk), **{c: v.ravel() for c, v in t.items()}}))
    fig, axs = plt.subplots(1, 4, figsize=(19, 4))
    for ax, (c, cmap, v) in zip(axs, [("weight_align", "viridis", None), ("write_auc", "RdBu_r", 0.5), ("to_question_auc", "RdBu_r", 0.5), ("entropy_auc", "RdBu_r", 0.5)]):
        M, kw = t[c], {}
        if v is not None:                                                                    # AUC maps: centred on 0.5
            r_ = max(0.05, float(np.nanmax(np.abs(M - v)))); kw = dict(vmin=v - r_, vmax=v + r_)
        im = ax.imshow(M, aspect="auto", origin="lower", cmap=cmap, **kw); plt.colorbar(im, ax=ax)
        ax.set_xlabel("head"); ax.set_ylabel("layer"); ax.set_title({"weight_align": "output weights along the direction", "write_auc": "writes along the direction: AUC",
                                                                     "to_question_auc": "attention to the question: AUC", "entropy_auc": "attention entropy: AUC"}[c], fontsize=9)
    fig.suptitle(f"{NAME[k]}: attention heads vs hallucinated entities (AUC > 0.5 = higher for hallucinated)"); savefig(f"6_heads_{k}")
    # correlation matrix (Spearman, over entities): the label, the direction, the top writing heads, the top attention heads
    feats = {"hallucinated": y, f"direction @ L{H['L']}": np.array([sp["proj"] for sp in H["spans"]])}
    feats.update({f"write L{l}H{h}": Cn[:, l, h] for l, h in H["top_heads"]})
    for j, st in enumerate(H["stats"]):                                                      # for each attention statistic, its 2 most telling heads
        for x in np.argsort(-np.abs(t[f"{st}_auc"] - 0.5).ravel())[:2]:
            l, h = int(x // NHk), int(x % NHk); feats[f"{st} L{l}H{h}"] = An[:, l, h, j]
    cm = pd.DataFrame(feats).corr(method="spearman"); cm.to_csv(f"{OUT}/correlation_{k}.csv")
    dirs = torch.load(fetch(f"{k}/direction.pt"), weights_only=False)["all"].float(); dirs = dirs / dirs.norm(dim=1, keepdim=True)
    fig, axs = plt.subplots(1, 2, figsize=(14, 5.6), gridspec_kw={"width_ratios": [1.3, 1]})
    im = axs[0].imshow(cm.values, cmap="RdBu_r", vmin=-1, vmax=1); plt.colorbar(im, ax=axs[0])
    axs[0].set_xticks(range(len(cm))); axs[0].set_xticklabels(cm.columns, rotation=70, fontsize=7); axs[0].set_yticks(range(len(cm))); axs[0].set_yticklabels(cm.columns, fontsize=7)
    axs[0].set_title("Spearman correlation over judged entities", fontsize=9)
    im = axs[1].imshow((dirs @ dirs.T).numpy(), cmap="RdBu_r", vmin=-1, vmax=1, origin="lower"); plt.colorbar(im, ax=axs[1])
    axs[1].set_xlabel("layer"); axs[1].set_ylabel("layer"); axs[1].set_title("cosine of the direction across layers", fontsize=9)
    fig.suptitle(NAME[k]); savefig(f"7_correlation_{k}")
    # attention patterns of the top heads around the first hallucinated entity (shaded: the entity)
    for ex in H["examples"][:1]:
        fig, axs = plt.subplots(1, len(ex["patterns"]), figsize=(3.2 * len(ex["patterns"]), 3.4), squeeze=False)
        for ax, (name, P) in zip(axs[0], ex["patterns"].items()):
            ax.imshow(P.float().numpy(), cmap="magma", vmin=0)
            for v in ex["span"]:
                ax.axhline(v, color="c", lw=0.6); ax.axvline(v, color="c", lw=0.6)
            ax.set_title(name, fontsize=9); ax.set_xlabel("key position"); ax.set_ylabel("query position")
        fig.suptitle(f"{NAME[k]} · {ex['id']}: top heads' attention around a hallucinated entity (lines = entity)", fontsize=10); savefig(f"8_attention_{k}")
if tabs:
    pd.concat(tabs).to_csv(f"{OUT}/head_stats.csv", index=False)

for f in ["detector_auc.csv", "per_answer.csv", "behaviour.csv", "head_stats.csv"] + [f"correlation_{k}.csv" for k in MODELS] + [f"figs/{x}" for x in sorted(os.listdir(f"{OUT}/figs"))]:
    if os.path.exists(f"{OUT}/{f}"):
        push(f)
log(f"results: https://huggingface.co/{OUT_REPO}/tree/main/{RUN}")